# Étape 3 — DE CUDA v1 (issues #9 et #10)

**Prêt à exécuter : Exécution → Tout exécuter.** Runtime : **GPU T4** (déjà réglé dans le notebook).

Le code est dans `src/cuda/de_cuda_v1.cu` et `src/cuda/cuda_utils.h`. Ce notebook le compile, vérifie les critères des deux issues et pousse un rapport sur la branche.

## Ce que fait la v1

Même algorithme que le DE séquentiel (DE/rand/1/bin, F = 0,5, CR = 0,3), mêmes fonctions (`benchmarks.h`), même budget 10⁴·D, même CSV.

| Kernel | Grille | Rôle |
|---|---|---|
| `k_init_rng` | (N+T−1)/T blocs, 1 thread par individu | un générateur cuRAND (Philox) par individu, graine du run |
| `k_init_pop` | idem | population uniforme dans le domaine |
| `k_eval` | idem | `fit[i] = evaluate(f, x_i, o, D)` — la fonction `__host__ __device__` de l'interface |
| `k_mutation_croisement` | idem | `trial[i]` = croisement de x_i et de x_r1 + F (x_r2 − x_r3), bornes respectées |
| `k_selection` | idem | si f(trial_i) ≤ f(x_i), x_i ← trial_i |

- **Mémoire** : `pop`, `trial`, `fit`, `fit_trial` et les générateurs en mémoire globale ; le décalage `o` en **mémoire constante** (tous les threads lisent `o[j]` au même moment : une seule lecture diffusée à tout le warp).
- **Aucun transfert dans la boucle** : une seule copie GPU → CPU à la fin (les N fitness).
- **Pas de race condition** : les essais sont écrits dans `trial`, la sélection n'a lieu qu'après, dans un autre kernel.
- **Erreurs** : `CUDA_CHECK` sur chaque appel CUDA, `CUDA_CHECK_KERNEL()` (`cudaGetLastError`) après chaque lancement.
- **Chronométrage** : `cudaFree(0)` crée le contexte avant le chrono, puis `cudaEvent` autour de tout l'algorithme (initialisation comprise, comme le séquentiel).

## Critères de validation
| Issue | Critère |
|---|---|
| #9 | même graine → même résultat ; erreur claire sur un lancement volontairement faux |
| #10 | compile sans warning avec `-arch=sm_75` ; écart de temps < 10 % entre deux runs identiques |

In [ ]:
from google.colab import userdata
GH_USER = "amenacer"
REPO    = "DE_CUDA"
BRANCHE = "amina/3.1-3.5-ecrire-de-cuda-v1"
token   = userdata.get("GITHUB_TOKEN")
!rm -rf /content/{REPO}
!git clone -q https://{token}@github.com/{GH_USER}/{REPO}.git /content/{REPO}
%cd /content/DE_CUDA
!git config user.name  "Amina Menacer"
!git config user.email "aminamenacer9@gmail.com"
!git checkout -q {BRANCHE} && git branch --show-current
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv

In [ ]:
import subprocess, re, statistics as st, os, json
ROOT = "/content/DE_CUDA"
RAPPORT = []                       # tout ce qui est affiche est aussi garde pour le rapport
def note(s=""):
    print(s); RAPPORT.append(str(s))

def run(args, cwd=ROOT, ok_codes=(0,)):
    p = subprocess.run(args, cwd=cwd, capture_output=True, text=True)
    if p.returncode not in ok_codes:
        print(p.stdout); print(p.stderr)
        raise RuntimeError(f"{args[0]} a renvoye {p.returncode}")
    return p

def compile_cuda(src, out):
    p = run(["nvcc", "-O3", "-arch=sm_75", "-Xcompiler", "-Wall", "-Xcompiler", "-Wextra",
             "-o", out, src], cwd=f"{ROOT}/src/cuda")
    warn = (p.stdout + p.stderr).strip()
    note(f"{src} : " + ("compilation SANS aucun warning" if not warn else "WARNINGS :\n" + warn))

KV = re.compile(r"(\w+)=(\S+)")
def lance(prog, f, D, N, seed, csv=None, extra=()):
    """Lance un run et renvoie le dictionnaire de sa derniere ligne (best_error, time_s, ...)."""
    args = [prog, f, str(D), str(N), str(seed)] + ([csv] if csv else []) + list(extra)
    out = run(args).stdout.strip().splitlines()[-1]
    d = dict(KV.findall(out))
    for k in ("best_error", "time_s"):
        d[k] = float(d[k])
    return d

def mann_whitney(a, b):
    """p-valeur du test de Mann-Whitney bilateral ; 1.0 si les deux series sont identiques et constantes."""
    from scipy.stats import mannwhitneyu
    if len(set(a) | set(b)) == 1:
        return 1.0
    return float(mannwhitneyu(a, b, alternative="two-sided").pvalue)

def tableau_md(lignes, entetes):
    s = "| " + " | ".join(entetes) + " |\n|" + "---|" * len(entetes) + "\n"
    for l in lignes:
        s += "| " + " | ".join(str(x) for x in l) + " |\n"
    return s
print("outils prets")

### 1 — Compiler (critère #10 : aucun warning, avec `-Wall -Wextra`)

In [ ]:
compile_cuda("de_cuda_v1.cu", "de_cuda_v1")
V1 = f"{ROOT}/src/cuda/de_cuda_v1"

### 2 — Lancement volontairement faux (critère #9)

`--test-erreur` lance un kernel avec 2048 threads par bloc (le maximum est 1024). `CUDA_CHECK_KERNEL` doit arrêter le programme avec un message clair, le fichier et la ligne.

In [ ]:
p = run([V1, "--test-erreur"], ok_codes=(0, 1))
note(p.stdout.strip()); note(p.stderr.strip())
note(f"code de retour : {p.returncode}  =>  " + ("erreur detectee : OK" if p.returncode == 1 and "invalid configuration" in p.stderr else "ECHEC"))

### 3 — Même graine → même résultat (critère #9)

In [ ]:
note("Determinisme (D = 10, N = 50)")
for f in ["sphere", "rastrigin", "rosenbrock", "griewank"]:
    a = lance(V1, f, 10, 50, 7); b = lance(V1, f, 10, 50, 7); c = lance(V1, f, 10, 50, 8)
    same = a["best_error"] == b["best_error"]
    note(f"  {f:10s} seed 7 : {a['best_error']:.15e} | seed 7 : {b['best_error']:.15e} "
         f"-> {'IDENTIQUES' if same else 'DIFFERENTS'} | seed 8 : {c['best_error']:.6e}")

### 4 — Stabilité du chronométrage (critère #10 : écart < 10 % entre deux runs identiques)

Rastrigin, D = 100, N = 100, même graine, 5 runs. On compare les deux premiers runs, puis l'écart maximal sur les 5.

In [ ]:
t = [lance(V1, "rastrigin", 100, 100, 1)["time_s"] for _ in range(5)]
e2 = abs(t[1] - t[0]) / min(t[0], t[1]) * 100
emax = (max(t) - min(t)) / min(t) * 100
note("Temps (s) de 5 runs identiques : " + ", ".join(f"{x:.4f}" for x in t))
note(f"ecart run1/run2 = {e2:.2f} %   ecart max/min sur 5 runs = {emax:.2f} %  =>  " + ("OK (< 10 %)" if e2 < 10 else "ECHEC"))

### 5 — Les 4 fonctions, D = 10 / 50 / 100, N = 100 : tout doit finir avec `sante=OK` et FE = 10⁴·D

In [ ]:
lignes = []
for f in ["sphere", "rosenbrock", "griewank", "rastrigin"]:
    for D in (10, 50, 100):
        r = lance(V1, f, D, 100, 1)
        lignes.append([f, D, 100, f"{r['best_error']:.3e}", r["FE"], f"{r['time_s']:.4f}", r["sante"]])
note(tableau_md(lignes, ["fonction", "D", "N", "best_error", "FE", "temps (s)", "sante"]))

### 6 — Sauvegarder le rapport et pousser sur la branche

In [ ]:
os.makedirs(f"{ROOT}/results", exist_ok=True)
with open(f"{ROOT}/results/etape3_v1_controles.md", "w") as fp:
    fp.write("# Étape 3 — contrôles du DE CUDA v1 (issues #9 et #10)\n\n")
    fp.write("GPU : " + run(["nvidia-smi", "--query-gpu=name", "--format=csv,noheader"]).stdout.strip() + "\n\n")
    fp.write("```\n" + "\n".join(RAPPORT) + "\n```\n")
%cd /content/DE_CUDA
!git add  results/etape3_v1_controles.md
!git status --short
!git commit -q -m "3.1-3.5 : controles du DE CUDA v1 sur GPU T4 (refs #9 #10)" && git push -q origin {BRANCHE} && echo "=> pousse sur {BRANCHE}"